In [117]:
import torch
from predictors.goal_probability_predictor import WorldCupGoalsPredictor
import json
from typing import List
import pickle
import pandas as pd

In [116]:
with open("models/4_goals_softmax/configs.json", "r") as f:
    cfg = json.load(f)

with open("preprocessed_data/2026_featurised_squads.json", "r") as f:
    squad_features = json.load(f)

model = WorldCupGoalsPredictor(cfg)
state_dict = torch.load("models/4_goals_softmax/model.pt", map_location="cpu")
model.load_state_dict(state_dict)
model.eval()

scaler = pickle.load(open("models/4_goals_softmax/scalers.pkl", "rb"))

In [120]:
def featurise(team1: str, team2: str, feature_order_list: List[str], isknockout: int=0) -> List[float]:
    team_1_features = squad_features[team1]
    team_2_features = squad_features[team2]
    features = {}
    for f in feature_order_list:
        if f == "is_knockout":
            features[f] = float(isknockout)
            continue

        if "team_1" in f:
            features[f] = float(team_1_features[f.replace("team_1_", "")])
        elif "team_2" in f:
            features[f] = float(team_2_features[f.replace("team_2_", "")])
        else:
            raise NotImplementedError(f"featurise function needs to be updated to handle feature {f}")

    return features
        

def predict_goals(team1, team2, isknockout=0, temperature=0.8):
    team_1 = featurise(team1, team2, cfg["model"]["features"], isknockout)
    team_2 = featurise(team2, team1, cfg["model"]["features"], isknockout)
    data = pd.DataFrame([team_1, team_2])
    data_scaled = scaler.transform(data)
    data_tensor = torch.tensor(data_scaled, dtype=torch.float32)
    with torch.no_grad():
        res = model.predict_with_temperature(data_tensor, temperature=temperature)
    return res


In [ ]:
predictions = {}
for i in range(100):
    res = predict_goals("Brazil", "Curacao", isknockout=0, temperature=1)
    res_tuple = (int(res[0]), int(res[1]))
    if res_tuple not in predictions:
        predictions[res_tuple] = 1
    else:
        predictions[res_tuple] += 1


{(1, 0): 27,
 (0, 2): 1,
 (3, 0): 15,
 (0, 0): 20,
 (0, 1): 6,
 (2, 0): 14,
 (4, 0): 2,
 (4, 1): 2,
 (1, 1): 5,
 (3, 1): 4,
 (1, 2): 2,
 (2, 2): 2}

In [4]:
model.input_dim

31